# Suite LOG — The logical layout

`Logical` defines a diagram's logical layout: diagrams, shapes and points, each owned, vectors and rotations, with no coordinates, and resolves the compass of a point or a shape: mbse-basis [CAP-237: logical layout schemas](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-237.md), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
import { Builders, JSON as Json, Paths, Validators } from "@mbse/schemas/Framework";

import { Logical } from "../src/Framework/index.js";

function assert(condition: unknown, message = "assertion failed"): asserts condition {
  if (!condition) throw new Error(`AssertionError: ${message}`);
}

const NAMES = ["Logical.OfRotation", "Logical.InDiagrams", "Logical.DiagramDirectory", "Logical.OfDiagram",
  "Logical.InDiagram", "Logical.OfShape", "Logical.Containment", "Logical.Adjacency", "Logical.OfPoint",
  "Logical.Origin", "Logical.Borders", "Logical.Extents", "Logical.Vector"];
const store: any = new Builders.Store();
store.transaction(Logical.install);
const L = store.Logical;
const diagrams = store.singleton("Logical.Diagrams");
const problems = (schema: any, root: any): string[] => Validators.Validate(store).Reachable(schema, root);
const same = (a: unknown, b: unknown): boolean => JSON.stringify(a) === JSON.stringify(b);

/** An object's path, as JSON text. */
function path(x: any): string {
  return Json.ToJSON(store)(store.Paths.OfRoot.as_schema(), Paths.of(x));
}

function turn(degrees: number): (r: any) => any {
  return (r: any) => r.degrees(degrees);
}

function origin(s: any): any {
  return s.origin((a: any) => a.insert((e: any) => e.point((p: any) => p)));
}

function named(name: string): (a: any) => any {
  return (a: any) => a.insert((e: any) => e.name(name).point((p: any) => p));
}

function vector(start: any, end: any, degrees: number): void {
  L.OfPoint(start, (p: any) => p.outgoing((a: any) => a.insert((e: any) => e.end(end).rotation(turn(degrees)))));
}

function raises(message: string, block: () => unknown): void {
  try {
    store.transaction(block);
  } catch (error) {
    assert(error instanceof Error && error.name === "ValueError", String(error));
    assert(error.message === message, error.message);
    return;
  }
  throw new Error(`AssertionError: expected ValueError: ${message}`);
}

store.transaction(() => L.DiagramDirectory(diagrams, (f: any) => f.diagrams((a: any) => a.insert((e: any) => e.name("overview").diagram(
  (d: any) => d.shapes((s: any) => s.insert((e: any) => e.name("cpu").shape((c: any) => origin(c)
    .borders(named("ne")).extents(named("e"))
    .children((k: any) => k.insert((e: any) => e.name("alu").child((x: any) => origin(x).borders(named("n")))))))))))));
const overview = diagrams.diagrams({ name: "overview" });
const cpu = overview.shapes({ name: "cpu" });
const alu = cpu.children({ name: "alu" });
const [o] = [...cpu.origin].map((e: any) => e.point);
const [ne, east] = [cpu.borders({ name: "ne" }), cpu.extents({ name: "e" })];

<a id="LOG-01"></a>
## LOG-01 · The install defines the logical layout's schemas, each valid, and run again writes nothing

In [ ]:
const schemas = NAMES.map((name) => store.registered(name));
assert(schemas.every((schema: any) => schema.validate().length === 0));
store.transaction(Logical.install);
assert(NAMES.every((name, i) => store.registered(name) === schemas[i]));
assert(diagrams === store.Instances.Logical.Diagrams);

<a id="LOG-02"></a>
## LOG-02 · Diagrams, shapes and points are owned: each found by its key, its path read back to Logical.Diagrams

In [ ]:
const one = (items: Iterable<any>, link: string, expected: any): boolean => {
  const found = [...items].map((e: any) => e[link]);
  return found.length === 1 && found[0] === expected;
};
assert(one(diagrams.diagrams, "diagram", overview) && one(overview.shapes, "shape", cpu));
assert(one(cpu.children, "child", alu) && one(alu.container, "container", cpu));
assert(one(o.origin_of, "shape", cpu) && one(ne.border_of, "shape", cpu));
assert(one(east.extent_of, "shape", cpu));
assert(path(alu.borders({ name: "n" })) === (
  '{"singleton": "Logical.Diagrams", "next": {"entry": {"i": "diagrams", "o": "directory", "properties"' +
  ': {"name": "overview"}, "next": {"entry": {"i": "shapes", "o": "diagram", "properties": {"name": "cp' +
  'u"}, "next": {"entry": {"i": "children", "o": "container", "properties": {"name": "alu"}, "next": {"' +
  'entry": {"i": "borders", "o": "border_of", "properties": {"name": "n"}, "next": {"target": {}}}}}}}}' +
  '}}}'));
assert(problems(L.OfDiagram, overview).length === 0);
raises("a 'Logical.OfPoint' needs an owner through 'origin_of', 'border_of', 'extent_of': give it one, since no " +
  "singleton can be", () => L.OfPoint((p: any) => p));
raises("a 'Logical.OfShape' needs an owner through 'diagram', 'container': give it one, since no singleton can be",
  () => L.OfShape((s: any) => s));

<a id="LOG-03"></a>
## LOG-03 · A shape has one origin, and a point one owner, as the validator reports

In [ ]:
store.transaction(() => L.OfShape(cpu, origin));
assert(same(problems(L.OfShape, cpu), ["unique(point) violated: entries agreeing on ['shape'] differ on ['point']"]));
const [extra] = [...cpu.origin].map((e: any) => e.point).filter((p: any) => p !== o);
store.transaction(() => L.OfShape(cpu, (s: any) => s.origin((a: any) => a.remove((e: any) => e.point(extra)))));
assert(one(cpu.origin, "point", o) && problems(L.OfShape, cpu).length === 0);
store.transaction(() => L.OfShape(alu, (s: any) => s.borders((a: any) => a.insert((e: any) => e.name("ne").point(ne)))));
assert(same(problems(L.OfShape, alu),
  ["unique(name, shape) violated: entries agreeing on ['point'] differ on ['name', 'shape']"]));
store.transaction(() => L.OfShape(alu, (s: any) => s.borders((a: any) => a.remove((e: any) => e.name("ne").point(ne)))));
assert(problems(L.OfDiagram, overview).length === 0);

<a id="LOG-04"></a>
## LOG-04 · A vector has one rotation per start and end

In [ ]:
store.transaction(() => vector(o, ne, 45));
const [v]: any[] = [...ne.incoming];
assert(v.start === o && v.end === ne && v.rotation.degrees.get() === 45 && problems(L.OfShape, cpu).length === 0);
store.transaction(() => vector(o, ne, 135));
assert(same(problems(L.OfShape, cpu),
  ["unique(rotation) violated: entries agreeing on ['end', 'start'] differ on ['rotation']"]));
store.transaction(() =>
  L.OfPoint(o, (p: any) => p.outgoing((a: any) => a.remove((e: any) => e.end(ne).rotation(turn(135))))));
assert(same([...ne.incoming].map((e: any) => e.rotation.degrees.get()), [45]) && problems(L.OfShape, cpu).length === 0);

<a id="LOG-05"></a>
## LOG-05 · A container's children, and adjacencies between siblings by rotation

In [ ]:
store.transaction(() => L.OfShape(cpu, (s: any) => s.children((k: any) => k.insert((e: any) => e.name("fpu").child(origin)))));
const fpu = cpu.children({ name: "fpu" });
store.transaction(() =>
  L.OfShape(alu, (s: any) => s.adjacent_to((a: any) => a.insert((e: any) => e.to_shape(fpu).rotation(turn(90))))));
const children = [...cpu.children].map((e: any) => e.child);
assert(children.length === 2 && children[0] === alu && children[1] === fpu);
const [adjacency]: any[] = [...fpu.adjacent_from];
assert(adjacency.from_shape === alu && adjacency.rotation.degrees.get() === 90 && problems(L.OfShape, cpu).length === 0);

<a id="LOG-06"></a>
## LOG-06 · A compass: a shape's and its origin's north, a border or extent point's its vector's rotation

In [ ]:
const n = alu.borders({ name: "n" });
store.transaction(() => {
  L.OfShape(cpu, (s: any) => s.extents(named("sw")));
  vector(o, east, -90);
  vector(o, cpu.extents({ name: "sw" }), 585.5);
  vector([...alu.origin][0].point, n, 0);
});
const sw = cpu.extents({ name: "sw" });
assert(Logical.compass(cpu) === 0 && Logical.compass(o) === 0 && Logical.compass(alu) === 0);
assert(Logical.compass(ne) === 45 && Logical.compass(east) === 270 && Logical.compass(sw) === 225.5 &&
  Logical.compass(n) === 0);

<a id="LOG-07"></a>
## LOG-07 · A compass the rules leave undefined is refused

In [ ]:
let w: any, south: any, bare: any;
store.transaction(() => {
  L.OfShape(cpu, (s: any) => s.borders(named("w")).borders(named("s")).children((k: any) => k.insert(
    (e: any) => e.name("bare").child((x: any) => x.borders(named("x"))))));
  L.OfShape(fpu, (s: any) => s.borders(named("q")));
  [w, south, bare] = [cpu.borders({ name: "w" }), cpu.borders({ name: "s" }), cpu.children({ name: "bare" })];
  L.OfPoint(o, (p: any) => p.outgoing((a: any) => a.insert((e: any) => e.end(south))));
  vector(o, w, 1);
  vector(o, w, 2);
});
raises("a shape without an origin has no compass", () => Logical.compass(bare));
raises("a shape without an origin has no compass", () => Logical.compass(bare.borders({ name: "x" })));
raises("a point with 0 vectors from its shape's origin, not one, has no compass",
  () => Logical.compass(fpu.borders({ name: "q" })));
raises("a point with 2 vectors from its shape's origin, not one, has no compass", () => Logical.compass(w));
raises("a point whose vector from its shape's origin has no rotation has no compass",
  () => Logical.compass(south));